In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
import os
load_dotenv()

In [ ]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

In [ ]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = model.invoke(messages)

    return {'messages':[response]}

In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

In [ ]:
thread_id = '1'
while True:
    user_msg = input("Type here: ")
    print('You: ', user_msg)

    if(user_msg.strip().lower() in ['exit', 'bye', 'quit', 'stop']):
        break
    
    config = {'configurable':{'thread_id': thread_id}};
    response = chatbot.invoke({'messages':[HumanMessage(content = user_msg)]}, config=config)

    print('AI: ', response['messages'][-1].content) 
